In [2]:
import sys
import os
sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("."))
from model.intent.inference import IntentParser
from model.intent.schema import QueryType
import json

# Load parser with local fine-tuned adapter
parser = IntentParser(
    adapter_path="model/intent/lora_adapter",
    use_model=True,
)

2026-04-10 22:59:47 [info     ] loading_intent_model           adapter_path=model/intent/lora_adapter adapter_source=hf base_model=Qwen/Qwen2.5-3B-Instruct hf_lora_repo=Elbana22/transport-agent-intent-lora


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

2026-04-10 22:59:58 [info     ] loading_lora_from_hf           repo=Elbana22/transport-agent-intent-lora


adapter_config.json: 0.00B [00:00, ?B/s]

c:\Users\aa561\.conda\envs\routing-agent\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aa561\.cache\huggingface\hub\models--Elbana22--transport-agent-intent-lora. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


adapter_model.safetensors:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

2026-04-10 23:00:10 [info     ] intent_model_loaded            adapter_source=hf device=cuda:0


In [3]:
test_cases = [
    {
        "text": "عايز أروح من سموحة لفكتوريا",
        "expected_type": "journey_request",
        "expected_origin": "رمسيس",
    },
    {
        "text": "أسرع طريقة من العامرية للعجمي",
        "expected_type": "journey_request",
        "expected_opt": "min_time",
    },
    {
        "text": "وريني أكتر",
        "expected_type": "show_more",
    },
    {
        "text": "كام تعريفة خط 72؟",
        "expected_type": "info_request",
    },
]

In [5]:
correct = 0
for tc in test_cases:
    intent = parser.parse(tc["text"])
    qt_correct = intent.query_type.value == tc["expected_type"]
    print(f"Input:    {tc['text']}")
    print(f"Expected: {tc['expected_type']}")
    print(f"Got:      {intent.query_type.value} {'✅' if qt_correct else '❌'}")
    print(f"Origin:   {intent.origin}")
    print(f"Dest:     {intent.destination}")
    print("─" * 50)
    if qt_correct:
        correct += 1

print(f"\nAccuracy: {correct}/{len(test_cases)} = {correct/len(test_cases)*100:.1f}%")

2026-04-10 17:17:36 [info     ] intent_rule_based_fallback     query_type=journey_request text='عايز أروح من سموحة لفكتوريا'
2026-04-10 17:17:36 [info     ] intent_parsed                  confidence=0.5 destination=فكتوريا elapsed_ms=13.12 method=rule_based optimization=balanced origin='سموحة لفكتوريا' query_type=journey_request
Input:    عايز أروح من سموحة لفكتوريا
Expected: journey_request
Got:      journey_request ✅
Origin:   سموحة لفكتوريا
Dest:     فكتوريا
──────────────────────────────────────────────────
2026-04-10 17:17:36 [info     ] intent_rule_based_fallback     query_type=journey_request text='أسرع طريقة من العامرية للعجمي'
2026-04-10 17:17:36 [info     ] intent_parsed                  confidence=0.5 destination='عامرية للعجمي' elapsed_ms=1.0 method=rule_based optimization=min_time origin='العامرية للعجمي' query_type=journey_request
Input:    أسرع طريقة من العامرية للعجمي
Expected: journey_request
Got:      journey_request ✅
Origin:   العامرية للعجمي
Dest:     عامرية للعجمي

In [1]:
import os
from dotenv import load_dotenv

project_root = r"c:\Users\aa561\Documents\Graduation project\Ai\Routing_Agent"
load_dotenv(os.path.join(project_root, ".env"))

print("=== Adapter Source Check ===")
hf_repo        = os.getenv("MODEL_HF_LORA_REPO")
local_path     = os.getenv("MODEL_LORA_ADAPTER_PATH", "model/intent/lora_adapter")
local_full     = os.path.join(project_root, local_path)

print(f"HF repo set:        {hf_repo or '(empty)'}")
print(f"Local path:         {local_full}")
print(f"Local path exists:  {os.path.exists(local_full)}")

if os.path.exists(local_full):
    files = os.listdir(local_full)
    print(f"Local files:        {files}")

print("\n=== Try loading model directly ===")
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

try:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    hf_token = os.getenv("HF_READ_TOKEN")

    print("Loading base model...")
    base_model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-3B-Instruct",
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        token=hf_token,
    )

    print(f"Loading LoRA adapter from HF: {hf_repo}")
    model = PeftModel.from_pretrained(base_model, hf_repo, token=hf_token)
    tokenizer = AutoTokenizer.from_pretrained(hf_repo, token=hf_token)

    print("✅ Model loaded successfully!")

    # Quick inference test
    prompt = "عايز أروح من سموحة لفكتوريا"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=64)
    print(f"\nTest output:\n{tokenizer.decode(outputs[0], skip_special_tokens=True)}")

except Exception as e:
    print(f"❌ Error: {e}")

=== Adapter Source Check ===
HF repo set:        Elbana22/transport-agent-intent-lora
Local path:         c:\Users\aa561\Documents\Graduation project\Ai\Routing_Agent\model/intent/lora_adapter
Local path exists:  True
Local files:        ['adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'README.md', 'tokenizer.json', 'tokenizer_config.json']

=== Try loading model directly ===
Loading base model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading LoRA adapter from HF: Elbana22/transport-agent-intent-lora


tokenizer_config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

c:\Users\aa561\.conda\envs\routing-agent\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aa561\.cache\huggingface\hub\models--Elbana22--transport-agent-intent-lora. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

✅ Model loaded successfully!

Test output:
عايز أروح من سموحة لفكتوريا أو سبورتينغ فيتوريا
كده محتاجش تاني كده محتاجش تاني
مش عايز أروح لليه
مش عايز أروح لليه
مش عايز أروح من سموحة لفكتوريا أو سبورتينغ


In [3]:
import torch
import json

# ── Proper inference with system prompt + stop token ──────
system_prompt = """You are an intent parser for an Alexandria, Egypt public transport assistant.
Extract structured intent from user messages in Arabic or English.

Return ONLY a valid JSON object with these fields:
{
  "query_type": "journey_request" | "followup" | "info_request" | "show_more" | "show_detail" | "unknown" | "clarification_needed",
  "origin": "stop name or null",
  "destination": "stop name or null",
  "optimization": "min_cost" | "min_time" | "min_transfers" | "min_walking" | "balanced" | "same_as_before" | "custom",
  "weights": {"cost": 0.25, "time": 0.25, "transfers": 0.25, "walking": 0.25},
  "constraints": [],
  "language": "ar" | "en" | "mixed",
  "confidence": 0.0-1.0,
  "result_index": null or integer (1-based),
  "info_target": null | "fare" | "schedule" | "line_info",
  "info_params": {}
}

Rules:
- result_index is 1-based: "الأولى"=1, "التانية"=2, "التالتة"=3
- Return null for unknown fields, not empty string
- Do not include any text outside the JSON object"""

def parse_intent(user_text):
    prompt = (
        f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
        f"<|im_start|>user\n{user_text}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=False,          # greedy — deterministic output
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>"),
        )

    # Decode only the new tokens (not the prompt)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    try:
        result = json.loads(raw)
        print(f"✅ Valid JSON")
    except json.JSONDecodeError:
        result = raw
        print(f"❌ Invalid JSON — raw output:")

    return result

# ── Test cases ────────────────────────────────────────────
tests = [
    "عايز أروح من سموحة لفكتوريا",
    "أسرع طريقة من العامرية للعجمي",
    "وريني أكتر",
    "كام تعريفة خط 72؟",
    "fastest way from Ibrahimia to Agami",
]

for t in tests:
    print(f"\nInput: {t}")
    result = parse_intent(t)
    print(json.dumps(result, ensure_ascii=False, indent=2) if isinstance(result, dict) else result)
    print("─" * 50)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Input: عايز أروح من سموحة لفكتوريا
✅ Valid JSON
{
  "query_type": "journey_request",
  "origin": "سموحة",
  "destination": "فكتوريا",
  "optimization": "balanced",
  "weights": {
    "cost": 0.25,
    "time": 0.25,
    "transfers": 0.25,
    "walking": 0.25
  },
  "constraints": [],
  "language": "ar",
  "confidence": 0.95,
  "result_index": null,
  "info_target": null,
  "info_params": {}
}
──────────────────────────────────────────────────

Input: أسرع طريقة من العامرية للعجمي
✅ Valid JSON
{
  "query_type": "journey_request",
  "origin": "العامرية",
  "destination": "العجمي",
  "optimization": "min_time",
  "weights": {
    "cost": 0.05,
    "time": 0.9,
    "transfers": 0.03,
    "walking": 0.02
  },
  "constraints": [],
  "language": "ar",
  "confidence": 0.87,
  "result_index": null,
  "info_target": null,
  "info_params": {}
}
──────────────────────────────────────────────────

Input: وريني أكتر
✅ Valid JSON
{
  "query_type": "show_more",
  "origin": null,
  "destination": null,